# Köppen-Geiger Results — Interactive

This page lets you **dial in the HBV calibration thresholds** with sliders and watch every plot below update live. Click **Live Code** in the top-right corner of the page to start the in-browser Python kernel — no install, no login, everything runs in your browser via Pyodide.

The data is loaded from a single bundled `all_results.json` produced by `preprocess_results.py` (run once, locally, against your `regions/` tree).

> **Calibration metric conventions**
> - `KGE`, `NSE` → higher is better (max = 1)
> - `Nelder-Mead` → objective value the optimizer minimised; lower is better


## 1. Setup

In [ ]:
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Display constants ─────────────────────────────────────────────────────
SCENARIO_ORDER = [
    "CMIP6 hist", "ERA5", "DestinE hist",
    "SSP1-2.6", "SSP2-4.5", "SSP3-7.0", "SSP5-8.5", "DestinE future",
]
KG_GROUP_COLOR = {
    "A": "#1565C0",  # Tropical    — blue
    "B": "#E65100",  # Arid        — deep orange
    "C": "#2E7D32",  # Temperate   — green
    "D": "#6A1B9A",  # Continental — purple
    "E": "#00838F",  # Polar       — teal
}
KG_GROUP_LABEL = {
    "A": "A — Tropical", "B": "B — Arid", "C": "C — Temperate",
    "D": "D — Continental", "E": "E — Polar",
}

# ── Calibration metric conventions ────────────────────────────────────────
# True  → higher value = better calibration (KGE, NSE)
# False → lower value = better calibration (Nelder-Mead objective)
CALIB_HIGHER_IS_BETTER = {"KGE": True, "NSE": True, "Nelder-Mead": False}


## 2. Load the bundled data

In a Pyodide environment (Live Code), `pyodide.http.pyfetch` downloads the JSON over HTTP. When running this notebook locally with a normal Jupyter kernel, we fall back to reading `all_results.json` from disk instead.

In [ ]:
DATA_URL = "all_results.json"   # served alongside the page in the Teachbook

async def _load_data():
    """Load all_results.json. Works both in Pyodide and a regular kernel."""
    try:
        # Pyodide path (Teachbooks Live Code)
        from pyodide.http import pyfetch
        resp = await pyfetch(DATA_URL)
        return await resp.json()
    except ImportError:
        # Local Jupyter path
        with open(DATA_URL) as f:
            return json.load(f)

# `await` works at the top level inside a Jupyter cell
rows = await _load_data()
df_full = (pd.DataFrame(rows)
             .rename(columns={"Nelder_Mead": "Nelder-Mead"})
             .sort_values(["kg_group", "kg_dominant", "region", "order"])
             .reset_index(drop=True))

n_regions = df_full["region"].nunique()
print(f"Loaded {len(df_full)} rows across {n_regions} regions.")


## 3. Dial in the calibration thresholds

Drag the sliders. Every plot and table below reacts immediately.

- **Min KGE** — drop regions with KGE below this value
- **Min NSE** — drop regions with NSE below this value
- **Max Nelder-Mead** — drop regions with NM objective above this value
- **Sort by** — which metric orders the surviving regions (best first)

In [ ]:
# ── Build the widgets ─────────────────────────────────────────────────────
# Slider ranges are pulled from the actual data so they always make sense.
kge_min, kge_max = float(df_full["KGE"].min()), float(df_full["KGE"].max())
nse_min, nse_max = float(df_full["NSE"].min()), float(df_full["NSE"].max())
nm_min,  nm_max  = float(df_full["Nelder-Mead"].min()), float(df_full["Nelder-Mead"].max())

w_kge = widgets.FloatSlider(value=kge_min, min=kge_min, max=kge_max, step=0.01,
                            description="Min KGE", continuous_update=False,
                            style={"description_width": "120px"},
                            layout=widgets.Layout(width="500px"))
w_nse = widgets.FloatSlider(value=nse_min, min=nse_min, max=nse_max, step=0.01,
                            description="Min NSE", continuous_update=False,
                            style={"description_width": "120px"},
                            layout=widgets.Layout(width="500px"))
w_nm  = widgets.FloatSlider(value=nm_max, min=nm_min, max=nm_max, step=0.01,
                            description="Max Nelder-Mead", continuous_update=False,
                            style={"description_width": "120px"},
                            layout=widgets.Layout(width="500px"))
w_sort = widgets.Dropdown(options=["KGE", "NSE", "Nelder-Mead"], value="KGE",
                          description="Sort by",
                          style={"description_width": "120px"})

controls = widgets.VBox([w_kge, w_nse, w_nm, w_sort])
display(controls)


## 4. Filter & sort

`apply_filter()` is the single function that turns the slider values into a filtered, sorted DataFrame. Every plot below calls it.

In [ ]:
def apply_filter(min_kge, min_nse, max_nm, sort_by):
    """Return (df_filtered, kept_regions_table) given the four control values."""
    calib = (df_full[["country", "region", "kg_dominant", "kg_group",
                       "KGE", "NSE", "Nelder-Mead"]]
              .drop_duplicates(subset="region")
              .reset_index(drop=True))

    mask = (calib["KGE"] >= min_kge) & (calib["NSE"] >= min_nse) & (calib["Nelder-Mead"] <= max_nm)
    kept = calib[mask].copy()

    ascending = not CALIB_HIGHER_IS_BETTER[sort_by]
    kept = kept.sort_values(sort_by, ascending=ascending).reset_index(drop=True)

    region_rank = {r: i for i, r in enumerate(kept["region"])}
    df = (df_full[df_full["region"].isin(region_rank)]
            .assign(_r=lambda d: d["region"].map(region_rank))
            .sort_values(["_r", "order"])
            .drop(columns="_r")
            .reset_index(drop=True))
    return df, kept


## 5. Live summary & plots

In [ ]:
# ── Plot helpers (one per output panel) ───────────────────────────────────
def _setup_axes(ax):
    ax.set_yscale("log")
    ax.axhline(100, color="black", linestyle="--", linewidth=1, alpha=0.4, zorder=1)
    ax.axvspan(-0.5, 2.5, color="#e8f4f8", alpha=0.4, zorder=0)
    ax.axvspan(2.5, len(SCENARIO_ORDER) - 0.5, color="#fff3e0", alpha=0.4, zorder=0)
    ax.text(1.0, 0.97, "Historical", transform=ax.get_xaxis_transform(),
            ha="center", va="top", color="#555", fontsize=9, style="italic")
    ax.text(5.0, 0.97, "Future", transform=ax.get_xaxis_transform(),
            ha="center", va="top", color="#555", fontsize=9, style="italic")


def plot_violin(df, ax):
    """Violin per scenario, points coloured by KG group."""
    _setup_axes(ax)
    datasets, positions = [], []
    for i, scen in enumerate(SCENARIO_ORDER):
        vals = df[df["scenario_label"] == scen]["rp_mean"].dropna().values
        if len(vals) > 0:
            datasets.append(vals); positions.append(i)
    if datasets:
        parts = ax.violinplot(datasets, positions=positions,
                              showmedians=True, showextrema=True, widths=0.6)
        for j, pc in enumerate(parts["bodies"]):
            color = "#1565C0" if positions[j] <= 2 else "#E65100"
            pc.set_facecolor(color); pc.set_alpha(0.35); pc.set_edgecolor(color)

    rng = np.random.default_rng(42)
    for i, scen in enumerate(SCENARIO_ORDER):
        sub = df[df["scenario_label"] == scen]
        jitter = rng.uniform(-0.08, 0.08, size=len(sub))
        for (_, row), jit in zip(sub.iterrows(), jitter):
            color = KG_GROUP_COLOR.get(row["kg_group"], "#888888")
            ax.scatter(i + jit, row["rp_mean"], color=color, s=30, zorder=4,
                       alpha=0.9, edgecolors="white", linewidths=0.4)

    ax.set_xticks(range(len(SCENARIO_ORDER)))
    ax.set_xticklabels(SCENARIO_ORDER, fontsize=10, rotation=10, ha="right")
    ax.set_ylabel("Return period of observed Q$_{100}$ (years, log scale)")
    ax.set_title(f"Return-period shift across scenarios — {df['region'].nunique()} regions")


def summary_table(df):
    """Mean RP per dominant KG class × scenario (with region count)."""
    if df.empty:
        return pd.DataFrame()
    pivot = df.pivot_table(index="kg_dominant", columns="scenario_label",
                            values="rp_mean", aggfunc="mean")
    cols = [c for c in SCENARIO_ORDER if c in pivot.columns]
    pivot = pivot[cols]
    counts = df.groupby("kg_dominant")["region"].nunique().rename("n_regions")
    pivot.insert(0, "n_regions", counts)
    pivot.index.name = "KG class (dominant)"
    return pivot


## 6. Live output

The cell below subscribes to all four sliders and re-renders on every change.

In [ ]:
out = widgets.Output()
display(out)


def _render(*_):
    df, kept = apply_filter(w_kge.value, w_nse.value, w_nm.value, w_sort.value)
    with out:
        clear_output(wait=True)

        # ── Top-line stats ────────────────────────────────────────────────
        n_total = df_full["region"].nunique()
        n_kept  = len(kept)
        ascending = not CALIB_HIGHER_IS_BETTER[w_sort.value]
        direction = "ascending (lower = better)" if ascending else "descending (higher = better)"
        print(f"Kept {n_kept} / {n_total} regions   |   "
              f"sorted by {w_sort.value} ({direction})")
        if n_kept == 0:
            print("\nNo regions pass the current thresholds. Loosen the sliders.")
            return

        # ── Calibration scores of the top 10 kept regions ─────────────────
        print("\nTop 10 regions by", w_sort.value + ":")
        top = kept.head(10)[["country", "region", "kg_dominant",
                              "KGE", "NSE", "Nelder-Mead"]]
        print(top.to_string(index=False,
                            formatters={"KGE":         lambda x: f"{x:6.3f}",
                                        "NSE":         lambda x: f"{x:6.3f}",
                                        "Nelder-Mead": lambda x: f"{x:6.3f}"}))

        # ── Violin plot ───────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(12, 5))
        plot_violin(df, ax)
        plt.tight_layout(); plt.show()

        # ── Pivot summary ─────────────────────────────────────────────────
        pivot = summary_table(df)
        if not pivot.empty:
            print("\nMean return period (years) per KG class × scenario:")
            print(pivot.round(1).to_string())


# Wire up + initial render
for w in (w_kge, w_nse, w_nm, w_sort):
    w.observe(_render, names="value")
_render()


## Notes

- **`Nelder-Mead` directionality**: this is the objective the optimizer minimised, so lower is better. If your convention is opposite, flip `CALIB_HIGHER_IS_BETTER["Nelder-Mead"]` in §1.
- **First load is slow**: Pyodide downloads ~30 MB of pandas + matplotlib the first time the page is opened. Subsequent visits are cached.
- **To regenerate the data** after new regions are added, re-run `preprocess_results.py` and commit the updated `all_results.json`.
